In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
df = spark.table("netflix.bronze.netflix_titles")


In [0]:
df = df.fillna({"duration_minutes":"0","duration_seasons":"0"})
df = df.withColumn("duration_minutes",F.col("duration_minutes").try_cast("int")).withColumn("duration_seasons",F.col("duration_seasons").try_cast("int"))
df = df.withColumn("rating",F.split(F.col("rating"),"-")[0]).withColumn("short_title",F.split(F.col("title"),":")[0])
df = df.withColumn("typeflag",F.when(F.col("type") == "Movie",1).when(F.col("type") == "TV Show",2))
df = df.withColumn("duration_ranking",F.dense_rank().over(Window.orderBy(F.col("duration_minutes").desc())))

df.write.format("delta").mode("overwrite").option("path","abfss://netflix@adls7786.dfs.core.windows.net/silver/netflix_titles").saveAsTable("netflix.silver.netflix_titles")